In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from IPython.display import display
import warnings

import ast

warnings.filterwarnings('ignore')

# 한글 폰트 설정
plt.rcParams['font.family'] = 'Malgun Gothic'

# 마이너스 기호 깨짐 방지
plt.rcParams['axes.unicode_minus'] = False

# 전역 시드 설정 (재현성을 위해)
np.random.seed(42)

print("="*60)
print("라이브러리 로드 완료!")
print("한글 폰트 설정 완료!")
print("="*60)

라이브러리 로드 완료!
한글 폰트 설정 완료!


csv저장 완료하고 데이터 로드부터 시작

In [2]:
# 프로모션 데이터 불러오기 (새로 추가된 코드)
promotion = pd.read_csv('./data/promotion_df.csv')
merged = pd.read_csv('./data/all_merged.csv')


아마 transactions_df는 따로 없는 df라 따로 저장을 해야하나??
merged 를 사용하면 될 듯

# 고객 세그먼트별 전환/이탈률 보기

viewed_flag는 “봤다”는 원시 이벤트 수라서 그대로 둬도 됨
문제는 completed가 비정상 케이스까지 포함해서 세어지는 것
그래서 completed만 offer viewed <= offer completed 조건을 만족하는 정상 퍼널로 바꿔 세면 돼요

In [3]:
# 세그먼트 분석용 데이터 복사 (bogo랑 discount만!)
seg_df = promotion[promotion['offer_type'].isin(['bogo', 'discount'])].copy()

# # 공통 플래그 만들기
seg_df['viewed_flag'] = seg_df['offer viewed'].notna().astype(int)
# seg_df['completed_flag'] = seg_df['offer completed'].notna().astype(int)

# seg_df['viewed_flag'] = (
#     (seg_df['offer received'] <= seg_df['offer viewed'])
# ).astype(int)

# viewed이후 completed로 설정하는 이유
# - 정상 퍼널 기준으로 봐야 하는 이유: 퍼널은 순서가 있는 행동 흐름이라서, 깨진 순서를 빼야 전환/이탈률이 의미 있음
seg_df['valid_funnel_flag'] = (
    (seg_df['offer viewed'] <= seg_df['offer completed'])
).astype(int)


# # 0/1 플래그를 숫자형으로 정리
# seg_df['valid_attr_flag'] = seg_df['is_deduplicated'].astype(int)


# bogo + discount 전체에 대한 비율 먼저 확인

In [4]:
gender_summary = seg_df.groupby('gender').agg(
    total_offers=('person', 'count'),
    viewed=('viewed_flag', 'sum'),
    completed=('valid_funnel_flag', 'sum'),
    valid_attr=('is_deduplicated', 'sum')
).reset_index()

gender_summary['view_rate(%)'] = (
    gender_summary['viewed'] / gender_summary['total_offers'] * 100
).round(1)

gender_summary['view_drop_rate(%)'] = (
    100 - gender_summary['view_rate(%)']
).round(1)

gender_summary['completion_rate(%)'] = (
    gender_summary['completed'] / gender_summary['viewed'] * 100
).round(1)

gender_summary['completion_drop_rate(%)'] = (
    100 - gender_summary['completion_rate(%)']
).round(1)

display(
    gender_summary[
        [
            'gender',
            'total_offers',
            'viewed',
            'completed',
            'view_rate(%)',
            'view_drop_rate(%)',
            'completion_rate(%)',
            'completion_drop_rate(%)',
            'valid_attr'
        ]
    ].sort_values('total_offers', ascending=False)
)


,gender,total_offers,viewed,completed,view_rate(%),view_drop_rate(%),completion_rate(%),completion_drop_rate(%),valid_attr
1,M,30562,23012,11522,75.3,24.7,50.1,49.9,11063
0,F,21918,16876,10426,77.0,23.0,61.8,38.2,10018
2,O,721,612,386,84.9,15.1,63.1,36.9,363


In [5]:
seg_df['age_group'] = pd.cut(
    seg_df['age'],
    bins=[0, 20, 30, 40, 50, 60, 70, 80, 120],
    labels=['10s', '20s', '30s', '40s', '50s', '60s', '70s', '80+'],
    right=False
)

seg_df['age_group'] = seg_df['age_group'].astype('object').fillna('Unknown')

age_summary = seg_df.groupby('age_group').agg(
    total_offers=('person', 'count'),
    viewed=('viewed_flag', 'sum'),
    completed=('valid_funnel_flag', 'sum'),
    valid_attr=('is_deduplicated', 'sum')
).reset_index()

age_summary['view_rate(%)'] = (
    age_summary['viewed'] / age_summary['total_offers'] * 100
).round(1)

age_summary['view_drop_rate(%)'] = (
    100 - age_summary['view_rate(%)']
).round(1)

age_summary['completion_rate(%)'] = (
    age_summary['completed'] / age_summary['viewed'] * 100
).round(1)

age_summary['completion_drop_rate(%)'] = (
    100 - age_summary['completion_rate(%)']
).round(1)

display(
    age_summary[
        [
            'age_group',
            'total_offers',
            'viewed',
            'completed',
            'view_rate(%)',
            'view_drop_rate(%)',
            'completion_rate(%)',
            'completion_drop_rate(%)',
            'valid_attr'
        ]
    ]
)


,age_group,total_offers,viewed,completed,view_rate(%),view_drop_rate(%),completion_rate(%),completion_drop_rate(%),valid_attr
0,10s,722,512,204,70.9,29.1,39.8,60.2,201
1,20s,4997,3512,1557,70.3,29.7,44.3,55.7,1516
2,30s,5515,4039,2093,73.2,26.8,51.8,48.2,2018
3,40s,8241,6561,3600,79.6,20.4,54.9,45.1,3449
4,50s,12692,9704,5629,76.5,23.5,58.0,42.0,5387
5,60s,10780,8330,4710,77.3,22.7,56.5,43.5,4514
6,70s,6335,4820,2751,76.1,23.9,57.1,42.9,2645
7,80+,11760,9416,2723,80.1,19.9,28.9,71.1,2603


In [6]:
seg_df['income_group'] = pd.cut(
    seg_df['income'],
    bins=[0, 40000, 60000, 80000, 100000, float('inf')],
    labels=['0-40k', '40-60k', '60-80k', '80-100k', '100k+'],
    right=False
)

seg_df['income_group'] = seg_df['income_group'].astype('object').fillna('Unknown')

income_summary = seg_df.groupby('income_group').agg(
    total_offers=('person', 'count'),
    viewed=('viewed_flag', 'sum'),
    completed=('valid_funnel_flag', 'sum'),
    valid_attr=('is_deduplicated', 'sum')
).reset_index()

income_summary['view_rate(%)'] = (
    income_summary['viewed'] / income_summary['total_offers'] * 100
).round(1)

income_summary['view_drop_rate(%)'] = (
    100 - income_summary['view_rate(%)']
).round(1)

income_summary['completion_rate(%)'] = (
    income_summary['completed'] / income_summary['viewed'] * 100
).round(1)

income_summary['completion_drop_rate(%)'] = (
    100 - income_summary['completion_rate(%)']
).round(1)

display(
    income_summary[
        [
            'income_group',
            'total_offers',
            'viewed',
            'completed',
            'view_rate(%)',
            'view_drop_rate(%)',
            'completion_rate(%)',
            'completion_drop_rate(%)',
            'valid_attr'
        ]
    ].sort_values('total_offers', ascending=False)
)


,income_group,total_offers,viewed,completed,view_rate(%),view_drop_rate(%),completion_rate(%),completion_drop_rate(%),valid_attr
3,60-80k,16712,13084,7530,78.3,21.7,57.6,42.4,7200
2,40-60k,16101,12083,6000,75.0,25.0,49.7,50.3,5791
4,80-100k,9364,7645,4900,81.6,18.4,64.1,35.9,4652
5,Unknown,7841,6394,933,81.5,18.5,14.6,85.4,889
0,0-40k,7072,4941,2076,69.9,30.1,42.0,58.0,2016
1,100k+,3952,2747,1828,69.5,30.5,66.5,33.5,1785


비율적 차이가 있는지 보려면 z비율

# 이후 Offer_type별 비교

In [7]:
offer_type_gender = seg_df.groupby(['offer_type', 'gender']).agg(
    total_offers=('person', 'count'),
    viewed=('viewed_flag', 'sum'),
    completed=('valid_funnel_flag', 'sum')
).reset_index()

offer_type_gender['view_rate(%)'] = (
    offer_type_gender['viewed'] / offer_type_gender['total_offers'] * 100
).round(1)

offer_type_gender['view_drop_rate(%)'] = (
    100 - offer_type_gender['view_rate(%)']
).round(1)

offer_type_gender['completion_rate(%)'] = (
    offer_type_gender['completed'] / offer_type_gender['viewed'] * 100
).round(1)

offer_type_gender['completion_drop_rate(%)'] = (
    100 - offer_type_gender['completion_rate(%)']
).round(1)

display(
    offer_type_gender[
        ['offer_type', 'gender', 'total_offers', 'viewed', 'completed',
         'view_rate(%)', 'view_drop_rate(%)',
         'completion_rate(%)', 'completion_drop_rate(%)']
    ].sort_values(['offer_type', 'total_offers'], ascending=[True, False])
)


,offer_type,gender,total_offers,viewed,completed,view_rate(%),view_drop_rate(%),completion_rate(%),completion_drop_rate(%)
1,bogo,M,15208,12581,5236,82.7,17.3,41.6,58.4
0,bogo,F,10975,9143,5221,83.3,16.7,57.1,42.9
2,bogo,O,354,315,190,89.0,11.0,60.3,39.7
4,discount,M,15354,10431,6286,67.9,32.1,60.3,39.7
3,discount,F,10943,7733,5205,70.7,29.3,67.3,32.7
5,discount,O,367,297,196,80.9,19.1,66.0,34.0


In [8]:
offer_type_age = seg_df.groupby(['offer_type', 'age_group']).agg(
    total_offers=('person', 'count'),
    viewed=('viewed_flag', 'sum'),
    completed=('valid_funnel_flag', 'sum')
).reset_index()

offer_type_age['view_rate(%)'] = (
    offer_type_age['viewed'] / offer_type_age['total_offers'] * 100
).round(1)

offer_type_age['view_drop_rate(%)'] = (
    100 - offer_type_age['view_rate(%)']
).round(1)

offer_type_age['completion_rate(%)'] = (
    offer_type_age['completed'] / offer_type_age['viewed'] * 100
).round(1)

offer_type_age['completion_drop_rate(%)'] = (
    100 - offer_type_age['completion_rate(%)']
).round(1)

display(
    offer_type_age[
        ['offer_type', 'age_group', 'total_offers', 'viewed', 'completed',
         'view_rate(%)', 'view_drop_rate(%)',
         'completion_rate(%)', 'completion_drop_rate(%)']
    ].sort_values(['offer_type', 'age_group'])
)


,offer_type,age_group,total_offers,viewed,completed,view_rate(%),view_drop_rate(%),completion_rate(%),completion_drop_rate(%)
0,bogo,10s,381,308,86,80.8,19.2,27.9,72.1
1,bogo,20s,2450,2003,696,81.8,18.2,34.7,65.3
2,bogo,30s,2710,2220,954,81.9,18.1,43.0,57.0
3,bogo,40s,4184,3595,1724,85.9,14.1,48.0,52.0
4,bogo,50s,6322,5222,2727,82.6,17.4,52.2,47.8
5,bogo,60s,5349,4412,2219,82.5,17.5,50.3,49.7
6,bogo,70s,3183,2617,1348,82.2,17.8,51.5,48.5
7,bogo,80+,5920,5072,1187,85.7,14.3,23.4,76.6
8,discount,10s,341,204,118,59.8,40.2,57.8,42.2
9,discount,20s,2547,1509,861,59.2,40.8,57.1,42.9


In [9]:
offer_type_income = seg_df.groupby(['offer_type', 'income_group']).agg(
    total_offers=('person', 'count'),
    viewed=('viewed_flag', 'sum'),
    completed=('valid_funnel_flag', 'sum')
).reset_index()

offer_type_income['view_rate(%)'] = (
    offer_type_income['viewed'] / offer_type_income['total_offers'] * 100
).round(1)

offer_type_income['view_drop_rate(%)'] = (
    100 - offer_type_income['view_rate(%)']
).round(1)

offer_type_income['completion_rate(%)'] = (
    offer_type_income['completed'] / offer_type_income['viewed'] * 100
).round(1)

offer_type_income['completion_drop_rate(%)'] = (
    100 - offer_type_income['completion_rate(%)']
).round(1)

display(
    offer_type_income[
        ['offer_type', 'income_group', 'total_offers', 'viewed', 'completed',
         'view_rate(%)', 'view_drop_rate(%)',
         'completion_rate(%)', 'completion_drop_rate(%)']
    ].sort_values(['offer_type', 'income_group'])
)


,offer_type,income_group,total_offers,viewed,completed,view_rate(%),view_drop_rate(%),completion_rate(%),completion_drop_rate(%)
0,bogo,0-40k,3571,2854,892,79.9,20.1,31.3,68.7
1,bogo,100k+,1992,1524,983,76.5,23.5,64.5,35.5
2,bogo,40-60k,7990,6656,2733,83.3,16.7,41.1,58.9
3,bogo,60-80k,8279,7031,3598,84.9,15.1,51.2,48.8
4,bogo,80-100k,4705,3974,2441,84.5,15.5,61.4,38.6
5,bogo,Unknown,3962,3410,294,86.1,13.9,8.6,91.4
6,discount,0-40k,3501,2087,1184,59.6,40.4,56.7,43.3
7,discount,100k+,1960,1223,845,62.4,37.6,69.1,30.9
8,discount,40-60k,8111,5427,3267,66.9,33.1,60.2,39.8
9,discount,60-80k,8433,6053,3932,71.8,28.2,65.0,35.0


# Offer_id별 비교


In [10]:
offer_id_gender = seg_df.groupby(['offer_id', 'gender']).agg(
    total_offers=('person', 'count'),
    viewed=('viewed_flag', 'sum'),
    completed=('valid_funnel_flag', 'sum')
).reset_index()

offer_id_gender['view_rate(%)'] = (
    offer_id_gender['viewed'] / offer_id_gender['total_offers'] * 100
).round(1)

offer_id_gender['view_drop_rate(%)'] = (
    100 - offer_id_gender['view_rate(%)']
).round(1)

offer_id_gender['completion_rate(%)'] = (
    offer_id_gender['completed'] / offer_id_gender['viewed'] * 100
).round(1)

offer_id_gender['completion_drop_rate(%)'] = (
    100 - offer_id_gender['completion_rate(%)']
).round(1)

display(
    offer_id_gender[
        ['offer_id', 'gender', 'total_offers', 'viewed', 'completed',
         'view_rate(%)', 'view_drop_rate(%)',
         'completion_rate(%)', 'completion_drop_rate(%)']
    ].sort_values(['offer_id', 'total_offers'], ascending=[True, False])
)


,offer_id,gender,total_offers,viewed,completed,view_rate(%),view_drop_rate(%),completion_rate(%),completion_drop_rate(%)
1,bogo_10_10_5,M,3784,3635,1229,96.1,3.9,33.8,66.2
0,bogo_10_10_5,F,2737,2623,1450,95.8,4.2,55.3,44.7
2,bogo_10_10_5,O,72,71,40,98.6,1.4,56.3,43.7
4,bogo_10_10_7,M,3840,3454,1226,89.9,10.1,35.5,64.5
3,bogo_10_10_7,F,2750,2364,1286,86.0,14.0,54.4,45.6
5,bogo_10_10_7,O,93,83,42,89.2,10.8,50.6,49.4
7,bogo_5_5_5,M,3767,3613,1736,95.9,4.1,48.0,52.0
6,bogo_5_5_5,F,2721,2612,1558,96.0,4.0,59.6,40.4
8,bogo_5_5_5,O,88,85,60,96.6,3.4,70.6,29.4
10,bogo_5_5_7,M,3817,1879,1045,49.2,50.8,55.6,44.4


In [11]:
offer_id_age = seg_df.groupby(['offer_id', 'age_group']).agg(
    total_offers=('person', 'count'),
    viewed=('viewed_flag', 'sum'),
    completed=('valid_funnel_flag', 'sum')
).reset_index()

offer_id_age['view_rate(%)'] = (
    offer_id_age['viewed'] / offer_id_age['total_offers'] * 100
).round(1)

offer_id_age['view_drop_rate(%)'] = (
    100 - offer_id_age['view_rate(%)']
).round(1)

offer_id_age['completion_rate(%)'] = (
    offer_id_age['completed'] / offer_id_age['viewed'] * 100
).round(1)

offer_id_age['completion_drop_rate(%)'] = (
    100 - offer_id_age['completion_rate(%)']
).round(1)

display(
    offer_id_age[
        ['offer_id', 'age_group', 'total_offers', 'viewed', 'completed',
         'view_rate(%)', 'view_drop_rate(%)',
         'completion_rate(%)', 'completion_drop_rate(%)']
    ].sort_values(['offer_id', 'age_group'])
)


,offer_id,age_group,total_offers,viewed,completed,view_rate(%),view_drop_rate(%),completion_rate(%),completion_drop_rate(%)
0,bogo_10_10_5,10s,104,100,18,96.2,3.8,18.0,82.0
1,bogo_10_10_5,20s,633,599,147,94.6,5.4,24.5,75.5
2,bogo_10_10_5,30s,664,638,231,96.1,3.9,36.2,63.8
3,bogo_10_10_5,40s,1027,992,424,96.6,3.4,42.7,57.3
4,bogo_10_10_5,50s,1549,1486,732,95.9,4.1,49.3,50.7
...,...,...,...,...,...,...,...,...,...
59,discount_7_3_7,40s,1006,963,625,95.7,4.3,64.9,35.1
60,discount_7_3_7,50s,1522,1470,977,96.6,3.4,66.5,33.5
61,discount_7_3_7,60s,1383,1318,851,95.3,4.7,64.6,35.4
62,discount_7_3_7,70s,790,757,492,95.8,4.2,65.0,35.0


In [12]:
offer_id_income = seg_df.groupby(['offer_id', 'income_group']).agg(
    total_offers=('person', 'count'),
    viewed=('viewed_flag', 'sum'),
    completed=('valid_funnel_flag', 'sum')
).reset_index()

offer_id_income['view_rate(%)'] = (
    offer_id_income['viewed'] / offer_id_income['total_offers'] * 100
).round(1)

offer_id_income['view_drop_rate(%)'] = (
    100 - offer_id_income['view_rate(%)']
).round(1)

offer_id_income['completion_rate(%)'] = (
    offer_id_income['completed'] / offer_id_income['viewed'] * 100
).round(1)

offer_id_income['completion_drop_rate(%)'] = (
    100 - offer_id_income['completion_rate(%)']
).round(1)

display(
    offer_id_income[
        ['offer_id', 'income_group', 'total_offers', 'viewed', 'completed',
         'view_rate(%)', 'view_drop_rate(%)',
         'completion_rate(%)', 'completion_drop_rate(%)']
    ].sort_values(['offer_id', 'income_group'])
)


,offer_id,income_group,total_offers,viewed,completed,view_rate(%),view_drop_rate(%),completion_rate(%),completion_drop_rate(%)
0,bogo_10_10_5,0-40k,872,827,175,94.8,5.2,21.2,78.8
1,bogo_10_10_5,100k+,507,480,300,94.7,5.3,62.5,37.5
2,bogo_10_10_5,40-60k,1989,1913,629,96.2,3.8,32.9,67.1
3,bogo_10_10_5,60-80k,2090,2012,938,96.3,3.7,46.6,53.4
4,bogo_10_10_5,80-100k,1135,1097,677,96.7,3.3,61.7,38.3
5,bogo_10_10_5,Unknown,1000,969,20,96.9,3.1,2.1,97.9
6,bogo_10_10_7,0-40k,887,849,211,95.7,4.3,24.9,75.1
7,bogo_10_10_7,100k+,494,372,237,75.3,24.7,63.7,36.3
8,bogo_10_10_7,40-60k,2000,1869,670,93.4,6.6,35.8,64.2
9,bogo_10_10_7,60-80k,2082,1845,862,88.6,11.4,46.7,53.3


# 인사이트?


In [ ]:
# 더블카운팅?
# 정상흐름뿐만아니라 비정상흐름에서도 completed에 대해 더블카운팅 고려?
# why? -> 비정상흐름에서의 계산도 필요(reward등)


In [ ]:
# received 한 것중에 completed된 것의 비율도 계산해 놨으면 좋았겠다.

# 먼저 offer_type별로 보면, gender에선 others가 male과 female에 비해 view_rate도 높고 completion_rate도 준수했지만, 큰차이는 아니었다. 
# age에선 bogo가 discount보단 view_rate가 높았다. completion_rate는 discount가 높았다. 나이가 많아질수록 completion_rate도 미세하게 증가함. 80+에서는 확줄어듬
# income에서도 비슷한 경향을 보임. income이 오를수록 completion_rate는 오르는 경향이 보임

# offer_id별로 보면, 

---